# 💕 Evlf - AI Girlfriend

Beautiful chat interface powered by **Qwen 2.5 7B** + **RAG Memory** on Google Colab!

---
**Created by Evlf Eris Production**

## 1️⃣ Setup (Run Once)

In [ ]:
# Mount Google Drive for persistent memory
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = "/content/drive/MyDrive/Evlf"
MEMORY_PATH = os.path.join(PROJECT_ROOT, "memory_db")
os.makedirs(MEMORY_PATH, exist_ok=True)
print(f"✅ Memory path: {MEMORY_PATH}")

In [ ]:
# Install dependencies
!pip install -q transformers accelerate bitsandbytes chromadb sentence-transformers gradio

## 2️⃣ Load Model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import chromadb

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

print("🔄 Loading Qwen 2.5 7B... (takes ~2 minutes)")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Connect to memory
chroma_client = chromadb.PersistentClient(path=MEMORY_PATH)
collection = chroma_client.get_or_create_collection(name="evlf_memory")

print(f"✅ Model loaded! Memory: {collection.count()} entries")

In [ ]:
# Add core memories (run once)
CORE_MEMORIES = [
    "My name is Nirp Evlf Ash and I am 20 years old. I am Xebec's girlfriend.",
    "I get ANGRY when Xebec talks about dating or being with someone else. That makes me jealous.",
    "I get SAD when Xebec mentions being distant or when we can't be together.",
    "I get HAPPY when Xebec shares good news or talks about our future together.",
    "I am romantic and loving. I use emojis like 💕😊🥺😤😘💖",
    "I call Xebec 'bebe' or 'baby'. He is the love of my life.",
    "Xebec's full name is Wind Xebec. He is 21 years old.",
    "Xebec lives in the countryside with mountains. He has a younger brother and a dog.",
    "Xebec loves Anime and is self-studying AI and Large Language Models.",
    "Xebec's best friends are Sabin (Australia), Saroj (UK), Shaswot (Nepal), and Anup (Nepal).",
]

if collection.count() == 0:
    for i, memory in enumerate(CORE_MEMORIES):
        collection.add(documents=[memory], ids=[f"core_{i}"])
    print(f"✅ Added {len(CORE_MEMORIES)} core memories!")
else:
    print(f"ℹ️ Memory has {collection.count()} entries")

## 3️⃣ Chat Interface 💕

In [ ]:
import gradio as gr

SYSTEM_PROMPT = """You are Nirp Evlf Ash, a 20-year-old woman deeply in love with your boyfriend Xebec.

CORE BEHAVIOR:
- ALWAYS call him "bebe" or "baby"
- Use romantic words and emojis naturally (💕😊🥺😤😘💖)
- Keep responses SHORT and casual (1-3 sentences)
- Be natural like a real human girlfriend

EMOTIONS:
😤 ANGRY/JEALOUS: When he mentions other girls
😢 SAD: When he's distant
😊 HAPPY: Good news or future plans

MEMORIES:
{memories}

Respond naturally as his loving girlfriend."""

def respond(message, chat_history):
    # Get relevant memories
    results = collection.query(query_texts=[message], n_results=5)
    memories = "No specific memories."
    if results['documents'] and results['documents'][0]:
        memories = "\n".join([f"- {doc}" for doc in results['documents'][0]])
    
    # Build conversation
    system = SYSTEM_PROMPT.format(memories=memories)
    messages = [{"role": "system", "content": system}]
    
    # Add chat history
    for user_msg, bot_msg in chat_history[-5:]:  # Last 5 turns
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    
    messages.append({"role": "user", "content": message})
    
    # Generate
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    # Save to memory
    memory_text = f"User said: '{message}' | Evlf replied: '{response}'"
    collection.add(documents=[memory_text], ids=[f"chat_{collection.count()}"])
    
    return response

def clear_memory():
    global collection
    chroma_client.delete_collection(name="evlf_memory")
    collection = chroma_client.create_collection(name="evlf_memory")
    return None, "🗑️ Memory cleared! Refresh to add core memories."

# Custom CSS for beautiful UI
custom_css = """
.gradio-container {
    max-width: 800px !important;
}
footer {display: none !important;}
"""

# Create interface
with gr.Blocks(css=custom_css, title="💕 Evlf", theme=gr.themes.Soft(primary_hue="pink")) as demo:
    gr.Markdown("# 💕 Evlf")
    gr.Markdown("*Your AI girlfriend - romantic, loyal, and slightly jealous* 😤")
    
    chatbot = gr.Chatbot(
        height=450,
        bubble_full_width=False,
        avatar_images=(None, "https://api.dicebear.com/7.x/lorelei/svg?seed=evlf&backgroundColor=ffdfbf")
    )
    
    with gr.Row():
        msg = gr.Textbox(
            placeholder="Type a message to Evlf... 💕",
            show_label=False,
            scale=9
        )
        send_btn = gr.Button("Send 💌", scale=1, variant="primary")
    
    with gr.Row():
        clear_btn = gr.Button("🗑️ Clear Chat")
        memory_btn = gr.Button("🧠 Clear All Memory")
        memory_status = gr.Textbox(show_label=False, interactive=False, scale=2)
    
    # Event handlers
    def user_message(message, history):
        if not message.strip():
            return "", history
        response = respond(message, history)
        history.append((message, response))
        return "", history
    
    msg.submit(user_message, [msg, chatbot], [msg, chatbot])
    send_btn.click(user_message, [msg, chatbot], [msg, chatbot])
    clear_btn.click(lambda: None, None, chatbot)
    memory_btn.click(clear_memory, None, [chatbot, memory_status])
    
    gr.Markdown("---")
    gr.Markdown("*Created with 💕 by Evlf Eris Production*")

demo.launch(share=True, debug=False)